# LoRA vs GPT-4o Comparison Evaluation

This notebook compares your fine-tuned LoRA model against GPT-4o on Monster Resort queries.

Metrics:
- **Response Quality**: Human evaluation (1-5 scale)
- **Latency**: Time to generate response
- **Cost**: OpenAI API costs
- **Availability**: Offline capability

In [ ]:
# Install dependencies
# !uv pip install torch transformers peft openai pandas matplotlib seaborn

import sys
sys.path.append('..')

import os
import time
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from openai import OpenAI

# Import your models
from test_lora import load_lora_model, generate_response

print('✅ Imports successful')

## 1. Load Models

In [ ]:
# Load LoRA model
print('Loading LoRA model...')
lora_model, lora_tokenizer = load_lora_model('lora-concierge/final')

# Initialize OpenAI
print('
Initializing OpenAI client...')
openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print('
✅ Both models ready!')

## 2. Define Test Queries

In [ ]:
test_queries = [
    {
        'query': 'What spa services are available at Castle Frankenstein?',
        'category': 'amenities',
        'difficulty': 'easy'
    },
    {
        'query': 'Tell me about Vampire Manor: Eternal Night Inn',
        'category': 'property_overview',
        'difficulty': 'easy'
    },
    {
        'query': 'What room types are available at Zombie Bed & Breakfast?',
        'category': 'rooms',
        'difficulty': 'easy'
    },
    {
        'query': 'What time is check-in across all properties?',
        'category': 'policy',
        'difficulty': 'medium'
    },
    {
        'query': 'Which properties are best for nocturnal guests?',
        'category': 'recommendation',
        'difficulty': 'medium'
    },
    {
        'query': 'Compare Vampire Manor and Werewolf Lodge',
        'category': 'comparison',
        'difficulty': 'hard'
    },
    {
        'query': 'What makes each property unique?',
        'category': 'comparison',
        'difficulty': 'hard'
    },
    {
        'query': 'Which property would you recommend for someone with claustrophobia?',
        'category': 'recommendation',
        'difficulty': 'hard'
    },
    {
        'query': 'What dining options are available?',
        'category': 'dining',
        'difficulty': 'medium'
    },
    {
        'query': 'Tell me about The Mummy Resort',
        'category': 'property_overview',
        'difficulty': 'easy'
    }
]

print(f'📋 {len(test_queries)} test queries prepared')

## 3. Run Comparison

In [ ]:
def generate_with_gpt4o(query: str) -> tuple[str, float]:
    """Generate response with GPT-4o and measure time."""
    start = time.time()
    
    response = openai_client.chat.completions.create(
        model='gpt-4o-mini',  # or 'gpt-4o'
        messages=[
            {
                'role': 'system',
                'content': 'You are the Monster Resort Concierge. Answer elegantly with gothic flair.'
            },
            {'role': 'user', 'content': query}
        ],
        temperature=0.7,
        max_tokens=200
    )
    
    latency = time.time() - start
    text = response.choices[0].message.content
    
    return text, latency


def generate_with_lora(query: str) -> tuple[str, float]:
    """Generate response with LoRA and measure time."""
    start = time.time()
    
    response = generate_response(
        lora_model,
        lora_tokenizer,
        query,
        max_new_tokens=200,
        temperature=0.7
    )
    
    latency = time.time() - start
    return response, latency


print('Running comparison...')
results = []

for i, test_case in enumerate(test_queries, 1):
    query = test_case['query']
    category = test_case['category']
    difficulty = test_case['difficulty']
    
    print(f'
[{i}/{len(test_queries)}] {query[:60]}...')
    
    # GPT-4o
    print('   GPT-4o...', end='', flush=True)
    gpt4o_response, gpt4o_latency = generate_with_gpt4o(query)
    print(f' ✓ {gpt4o_latency:.2f}s')
    
    # LoRA
    print('   LoRA...', end='', flush=True)
    lora_response, lora_latency = generate_with_lora(query)
    print(f' ✓ {lora_latency:.2f}s')
    
    results.append({
        'query': query,
        'category': category,
        'difficulty': difficulty,
        'gpt4o_response': gpt4o_response,
        'gpt4o_latency': gpt4o_latency,
        'lora_response': lora_response,
        'lora_latency': lora_latency
    })
    
    time.sleep(1)  # Rate limiting

print('
✅ Comparison complete!')

# Convert to DataFrame
df = pd.DataFrame(results)
df.head()

## 4. Latency Analysis

In [ ]:
# Calculate statistics
print('📊 Latency Statistics
')
print(f'GPT-4o:')
print(f'  Average: {df["gpt4o_latency"].mean():.2f}s')
print(f'  Min: {df["gpt4o_latency"].min():.2f}s')
print(f'  Max: {df["gpt4o_latency"].max():.2f}s')
print()
print(f'LoRA:')
print(f'  Average: {df["lora_latency"].mean():.2f}s')
print(f'  Min: {df["lora_latency"].min():.2f}s')
print(f'  Max: {df["lora_latency"].max():.2f}s')
print()
print(f'Speedup: {df["gpt4o_latency"].mean() / df["lora_latency"].mean():.2f}x')

In [ ]:
# Visualization
fig, ax = plt.subplots(figsize=(10, 6))

x = range(len(df))
width = 0.35

ax.bar([i - width/2 for i in x], df['gpt4o_latency'], width, label='GPT-4o', color='#3498db')
ax.bar([i + width/2 for i in x], df['lora_latency'], width, label='LoRA (Phi-3)', color='#e74c3c')

ax.set_xlabel('Query', fontsize=12)
ax.set_ylabel('Latency (seconds)', fontsize=12)
ax.set_title('Response Latency: GPT-4o vs LoRA Fine-tuned Phi-3', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'Q{i+1}' for i in x])
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../docs/lora_vs_gpt4o_latency.png', dpi=300, bbox_inches='tight')
plt.show()

print('✅ Chart saved to docs/lora_vs_gpt4o_latency.png')

## 5. Quality Comparison (Manual Review)

In [ ]:
# Display responses side-by-side for manual evaluation
for i, row in df.iterrows():
    print('=' * 80)
    print(f'Query {i+1}: {row["query"]}')
    print('=' * 80)
    print('
GPT-4o Response:')
    print(row['gpt4o_response'])
    print('
LoRA Response:')
    print(row['lora_response'])
    print('
')

### Manual Quality Scores

Rate each response on a scale of 1-5:
- 1: Poor (inaccurate, off-topic)
- 2: Fair (somewhat relevant but lacks detail)
- 3: Good (accurate, adequate)
- 4: Very Good (accurate, detailed, well-written)
- 5: Excellent (perfect answer, great style)

In [ ]:
# Add your manual quality scores here
quality_scores = {
    'gpt4o': [4, 5, 4, 4, 5, 4, 5, 4, 4, 5],  # Example scores
    'lora': [3, 4, 3, 3, 4, 3, 4, 3, 3, 4]     # Example scores
}

df['gpt4o_quality'] = quality_scores['gpt4o']
df['lora_quality'] = quality_scores['lora']

print('📊 Quality Scores')
print(f'GPT-4o average: {df["gpt4o_quality"].mean():.2f}/5')
print(f'LoRA average: {df["lora_quality"].mean():.2f}/5')
print(f'Gap: {df["gpt4o_quality"].mean() - df["lora_quality"].mean():.2f} points')

## 6. Cost Analysis

In [ ]:
# GPT-4o-mini pricing (Jan 2026)
INPUT_COST_PER_1M = 0.150  # $0.150 per 1M input tokens
OUTPUT_COST_PER_1M = 0.600  # $0.600 per 1M output tokens

# Estimate tokens (rough: 1 token ~= 4 characters)
df['gpt4o_input_tokens'] = df['query'].str.len() / 4
df['gpt4o_output_tokens'] = df['gpt4o_response'].str.len() / 4

df['gpt4o_cost'] = (
    (df['gpt4o_input_tokens'] * INPUT_COST_PER_1M / 1_000_000) +
    (df['gpt4o_output_tokens'] * OUTPUT_COST_PER_1M / 1_000_000)
)

print('💰 Cost Analysis
')
print(f'GPT-4o Cost per Query: ${df["gpt4o_cost"].mean():.6f}')
print(f'LoRA Cost per Query: $0 (local inference)')
print()
print(f'At 1,000 queries/day:')
print(f'  GPT-4o: ${df["gpt4o_cost"].mean() * 1000:.2f}/day = ${df["gpt4o_cost"].mean() * 1000 * 365:.2f}/year')
print(f'  LoRA: $0/day = $0/year')
print()
print(f'💡 Annual Savings: ${df["gpt4o_cost"].mean() * 1000 * 365:.2f}')

## 7. Summary Table

In [ ]:
summary = pd.DataFrame({
    'Model': ['GPT-4o-mini', 'LoRA (Phi-3-mini)'],
    'Avg Latency (s)': [
        df['gpt4o_latency'].mean(),
        df['lora_latency'].mean()
    ],
    'Avg Quality (1-5)': [
        df['gpt4o_quality'].mean(),
        df['lora_quality'].mean()
    ],
    'Cost/Query': [
        f'${df["gpt4o_cost"].mean():.6f}',
        '$0'
    ],
    'Offline': ['❌', '✅'],
    'Use Case': [
        'High-quality responses',
        'Fallback / Cost-sensitive'
    ]
})

print('
📊 Model Comparison Summary')
print('=' * 80)
print(summary.to_string(index=False))
print('=' * 80)

## 8. Export Results

In [ ]:
# Save full results
df.to_json('../docs/lora_vs_gpt4o_results.json', orient='records', indent=2)

# Save summary
summary.to_csv('../docs/lora_vs_gpt4o_summary.csv', index=False)

print('✅ Results exported:')
print('   - docs/lora_vs_gpt4o_results.json')
print('   - docs/lora_vs_gpt4o_summary.csv')
print('   - docs/lora_vs_gpt4o_latency.png')

## 9. Conclusions

**When to use GPT-4o:**
- Maximum quality needed
- Complex reasoning required
- Cost is not a concern

**When to use LoRA:**
- OpenAI unavailable (outage, rate limits)
- Cost-sensitive scenarios (high volume)
- Offline capability required
- Acceptable quality for simple queries

**Recommendation:**
Use GPT-4o as primary, LoRA as fallback. This provides best quality with resilience and cost control.